# RL 动态调参高频交易策略

## 论文参考

- **标题**: RL-based HFT Parameter Tuning
- **作者**: Weipeng Zhang
- **年份**: 2023
- **关键结果**: 平均利润 23,347 RMB

### 摘要

本文使用强化学习 (DQN) 动态调整布林带策略参数 (窗口期、标准差倍数)。
RL 智能体根据当前市场状态 (波动率、趋势强度) 选择最优参数组合，
使策略能自适应不同市场环境。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### DQN 动态调参

**1. 状态空间 (State)**

$$s_t = [\text{vol}_{20}, \text{trend}_{20}, \text{rsi}, \text{bb\_width}, \text{position}]$$

- $\text{vol}_{20}$: 20日波动率
- $\text{trend}_{20}$: 20日动量方向
- $\text{rsi}$: RSI 指标
- $\text{bb\_width}$: 当前布林带宽度
- $\text{position}$: 持仓状态

**2. 动作空间 (Action)**

$$a \in \{(w, k) : w \in \{10, 15, 20, 25, 30\}, k \in \{1.5, 2.0, 2.5, 3.0\}\}$$

20 种参数组合 (5 种窗口 x 4 种标准差倍数)。

**3. 奖励 (Reward)**

$$r_t = \text{strategy\_return}_t - \text{transaction\_cost}_t$$

**4. DQN 网络**

$$Q(s, a; \theta) \approx \mathbb{E}[R_t | s_t = s, a_t = a]$$

使用经验回放和目标网络稳定训练。

In [ ]:
# ============================================================
# Cell 3: 动画展示 - RL 智能体调参过程
# ============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

# 模拟RL训练过程: 参数选择随训练改善
n_episodes = 30
windows = [10, 15, 20, 25, 30]
stds = [1.5, 2.0, 2.5, 3.0]

# 假设真实最优参数随市场变化
episode_rewards = []
window_choices = []
std_choices = []

for ep in range(n_episodes):
    # 早期随机探索, 后期集中到好的参数
    explore_rate = max(0.1, 1.0 - ep / n_episodes)
    if np.random.random() < explore_rate:
        w = np.random.choice(windows)
        s = np.random.choice(stds)
    else:
        w = 20  # 模拟学到的最优
        s = 2.0
    window_choices.append(w)
    std_choices.append(s)

    # 奖励逐步增加
    base_reward = -0.5 + ep * 0.08
    noise = np.random.randn() * 0.3
    episode_rewards.append(base_reward + noise)

frames = []
for step in range(1, n_episodes + 1):
    frames.append(go.Frame(
        data=[
            go.Scatter(x=list(range(step)), y=episode_rewards[:step],
                       mode='lines+markers', name='Episode Reward',
                       line=dict(color='#F44336', width=2)),
            go.Scatter(x=list(range(step)), y=window_choices[:step],
                       mode='markers', name='Window选择',
                       marker=dict(color='#2196F3', size=8), yaxis='y2'),
            go.Scatter(x=list(range(step)),
                       y=[s * 10 for s in std_choices[:step]],  # 放大显示
                       mode='markers', name='StdMult*10',
                       marker=dict(color='#4CAF50', size=6, symbol='diamond'), yaxis='y2'),
        ],
        name=f'Ep {step}'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='DQN 动态调参: 智能体学习最优布林带参数'),
        xaxis_title='Episode',
        yaxis=dict(title='Episode Reward'),
        yaxis2=dict(title='参数值', overlaying='y', side='right'),
        height=500, width=900,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 300}}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import warnings
import random
import numpy as np
import pandas as pd
from collections import deque

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_etf_daily, get_index_daily
from shared.backtest_engine import Backtester
from shared.factors import ema, rsi, bollinger_bands
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_cumulative_comparison
)
from shared.constants import DEFAULT_START, DEFAULT_END, ETF_CSI300

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'模块导入完成, device={device}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================
try:
    df = get_etf_daily(ETF_CSI300, DEFAULT_START, DEFAULT_END)
    print(f'ETF 510300 数据: {len(df)} 条')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    n = len(dates)
    np.random.seed(42)
    price = 4.5 + np.cumsum(np.random.randn(n) * 0.015)
    price = np.maximum(price, 2.0)
    df = pd.DataFrame({
        'open': price * (1 + np.random.randn(n) * 0.003),
        'close': price,
        'high': price * (1 + np.abs(np.random.randn(n) * 0.008)),
        'low': price * (1 - np.abs(np.random.randn(n) * 0.008)),
        'volume': np.random.randint(50000, 2000000, n).astype(float),
    }, index=dates)

print(f'数据: {len(df)} 条, {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')

In [ ]:
# ============================================================
# Cell 6: 交易环境 + DQN 定义
# ============================================================

# 参数动作空间
WINDOWS = [10, 15, 20, 25, 30]
STD_MULTS = [1.5, 2.0, 2.5, 3.0]
ACTIONS = [(w, s) for w in WINDOWS for s in STD_MULTS]  # 20 actions
N_ACTIONS = len(ACTIONS)


class BollingerTuningEnv:
    """布林带参数调优环境"""

    def __init__(self, prices, lookback=30):
        self.prices = prices.values if hasattr(prices, 'values') else prices
        self.n = len(self.prices)
        self.lookback = lookback
        self.reset()

    def reset(self):
        self.t = self.lookback + 30  # 确保有足够历史
        self.position = 0
        self.current_window = 20
        self.current_std = 2.0
        return self._get_state()

    def _get_state(self):
        """构建状态向量"""
        hist = self.prices[max(0,self.t-60):self.t]
        rets = np.diff(hist) / hist[:-1] if len(hist) > 1 else [0]
        vol_20 = np.std(rets[-20:]) if len(rets) >= 20 else 0
        trend_20 = np.mean(rets[-20:]) if len(rets) >= 20 else 0
        # RSI 简单计算
        gains = [max(0, r) for r in rets[-14:]]
        losses = [max(0, -r) for r in rets[-14:]]
        avg_gain = np.mean(gains) if gains else 0
        avg_loss = np.mean(losses) if losses else 1e-8
        rsi_val = 100 - 100 / (1 + avg_gain / max(avg_loss, 1e-8))
        # 布林带宽度
        ma = np.mean(hist[-self.current_window:]) if len(hist) >= self.current_window else hist[-1]
        std = np.std(hist[-self.current_window:]) if len(hist) >= self.current_window else 0
        bb_width = std / ma if ma > 0 else 0

        return np.array([vol_20 * 100, trend_20 * 100, rsi_val / 100,
                         bb_width * 100, float(self.position)], dtype=np.float32)

    def step(self, action_idx):
        """执行动作: 设定新参数, 模拟5天交易, 返回奖励"""
        window, std_mult = ACTIONS[action_idx]
        self.current_window = window
        self.current_std = std_mult

        reward = 0
        steps = 5  # 每次动作持续5天

        for _ in range(steps):
            if self.t >= self.n - 1:
                return self._get_state(), reward, True

            # 布林带信号
            hist = self.prices[max(0, self.t-window):self.t]
            if len(hist) < window:
                self.t += 1
                continue
            ma = np.mean(hist)
            std = np.std(hist)
            upper = ma + std_mult * std
            lower = ma - std_mult * std
            price = self.prices[self.t]

            # 交易逻辑
            old_pos = self.position
            if self.position == 0 and price < lower:
                self.position = 1
            elif self.position == 1 and price > upper:
                self.position = 0

            # 收益计算
            ret = (self.prices[self.t] - self.prices[self.t-1]) / self.prices[self.t-1]
            if old_pos == 1:
                reward += ret
            # 交易成本
            if old_pos != self.position:
                reward -= 0.001

            self.t += 1

        done = self.t >= self.n - 1
        return self._get_state(), reward, done


class DQN(nn.Module):
    def __init__(self, state_dim=5, n_actions=20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)


print(f'动作空间: {N_ACTIONS} 种参数组合')
print(f'参数组合示例: {ACTIONS[:5]}...')

In [ ]:
# ============================================================
# Cell 7: DQN 训练
# ============================================================

# 超参数
N_EPISODES = 80
BATCH_SIZE = 32
GAMMA = 0.95
LR = 0.001
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 0.98
TARGET_UPDATE = 10
BUFFER_SIZE = 2000

# 初始化
policy_net = DQN(5, N_ACTIONS).to(device)
target_net = DQN(5, N_ACTIONS).to(device)
target_net.load_state_dict(policy_net.state_dict())
optimizer = optim.Adam(policy_net.parameters(), lr=LR)

replay_buffer = deque(maxlen=BUFFER_SIZE)
epsilon = EPS_START
episode_rewards_hist = []
param_choices_hist = []

env = BollingerTuningEnv(df['close'])

for ep in range(N_EPISODES):
    state = env.reset()
    total_reward = 0
    ep_params = []

    while True:
        # Epsilon-greedy
        if random.random() < epsilon:
            action = random.randrange(N_ACTIONS)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                action = policy_net(state_t).argmax(1).item()

        next_state, reward, done = env.step(action)
        replay_buffer.append((state, action, reward, next_state, done))
        total_reward += reward
        ep_params.append(ACTIONS[action])
        state = next_state

        # 训练
        if len(replay_buffer) >= BATCH_SIZE:
            batch = random.sample(replay_buffer, BATCH_SIZE)
            states, actions, rewards, next_states, dones = zip(*batch)

            states_t = torch.FloatTensor(np.array(states)).to(device)
            actions_t = torch.LongTensor(actions).to(device)
            rewards_t = torch.FloatTensor(rewards).to(device)
            next_states_t = torch.FloatTensor(np.array(next_states)).to(device)
            dones_t = torch.FloatTensor(dones).to(device)

            q_values = policy_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze()
            with torch.no_grad():
                next_q = target_net(next_states_t).max(1)[0]
                target_q = rewards_t + GAMMA * next_q * (1 - dones_t)

            loss = nn.MSELoss()(q_values, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break

    epsilon = max(EPS_END, epsilon * EPS_DECAY)
    episode_rewards_hist.append(total_reward)
    param_choices_hist.append(ep_params)

    if ep % 10 == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if ep % 20 == 0:
        print(f'Episode {ep}/{N_EPISODES}, Reward: {total_reward:.4f}, '
              f'Epsilon: {epsilon:.3f}')

print(f'\n训练完成! 最终 Reward: {episode_rewards_hist[-1]:.4f}')

In [ ]:
# ============================================================
# Cell 8: RL策略回测 vs 固定参数策略
# ============================================================

# RL 策略: 使用训练好的DQN选参数
env_test = BollingerTuningEnv(df['close'])
state = env_test.reset()

rl_signals = pd.Series(0, index=df.index)
t_start = env_test.t

while True:
    with torch.no_grad():
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        action = policy_net(state_t).argmax(1).item()
    old_t = env_test.t
    state, reward, done = env_test.step(action)
    # 记录持仓
    for t_idx in range(old_t, min(env_test.t, len(df))):
        rl_signals.iloc[t_idx] = env_test.position
    if done:
        break

# 固定参数布林带策略 (window=20, std=2.0)
bb = bollinger_bands(df['close'], 20, 2.0)
fixed_signals = pd.Series(0, index=df.index)
pos = 0
for i in range(1, len(df)):
    if pd.isna(bb['bb_lower'].iloc[i]):
        fixed_signals.iloc[i] = pos
        continue
    if pos == 0 and df['close'].iloc[i] < bb['bb_lower'].iloc[i]:
        pos = 1
    elif pos == 1 and df['close'].iloc[i] > bb['bb_upper'].iloc[i]:
        pos = 0
    fixed_signals.iloc[i] = pos

# 获取基准
try:
    bench = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
    bench_prices = bench['close']
except:
    bench_prices = None

backtester = Backtester(initial_capital=1_000_000, t_plus_1=True)

rl_result = backtester.run(df['close'], rl_signals, bench_prices)
fixed_result = backtester.run(df['close'], fixed_signals, bench_prices)

print('=== RL 动态调参策略 ===')
for k, v in rl_result['metrics'].items():
    print(f'  {k}: {v}')

print('\n=== 固定参数布林带 ===')
for k, v in fixed_result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 策略对比
strategies = {
    'RL 动态调参': rl_result['returns'],
    '固定参数 (20, 2.0)': fixed_result['returns'],
}
plot_cumulative_comparison(strategies, title='RL动态调参 vs 固定参数布林带')

# 2. 收益曲线
bench_eq = None
if rl_result.get('benchmark_returns') is not None:
    bench_eq = (1 + rl_result['benchmark_returns']).cumprod() * rl_result['equity_curve'].iloc[0]
plot_equity_curve(rl_result['equity_curve'], bench_eq,
                  title='RL 动态调参 - 累计收益')

# 3. 回撤
plot_drawdown(rl_result['equity_curve'], title='RL 动态调参 - 回撤')

# 4. 训练曲线 + 参数分布
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 训练奖励
ax1.plot(episode_rewards_hist, color='#F44336', alpha=0.5, linewidth=1)
ax1.plot(pd.Series(episode_rewards_hist).rolling(10).mean(),
         color='#F44336', linewidth=2.5, label='MA10')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.set_title('DQN 训练奖励曲线')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 最后一个episode的参数选择分布
if param_choices_hist:
    last_params = param_choices_hist[-1]
    windows_chosen = [p[0] for p in last_params]
    stds_chosen = [p[1] for p in last_params]
    ax2.hist(windows_chosen, bins=len(WINDOWS), color='#2196F3', alpha=0.7, label='Window')
    ax2.set_xlabel('Window 参数')
    ax2.set_ylabel('选择次数')
    ax2.set_title('最终Episode参数选择分布')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. 绩效对比表
comparison = {}
for k in rl_result['metrics']:
    comparison[k] = f"RL: {rl_result['metrics'][k]}  |  Fixed: {fixed_result['metrics'][k]}"
plot_metrics_table(comparison, title='RL vs 固定参数 绩效对比')

## 结果讨论

### 策略表现

1. **自适应**: RL 根据市场状态动态选择布林带参数
2. **探索利用**: Epsilon-greedy 在训练初期探索不同参数组合
3. **参数集中**: 训练后期智能体倾向于选择表现好的参数

### 局限性

- DQN 训练需要大量样本，日线数据量有限
- 离散动作空间限制了参数精度
- 奖励设计对策略表现影响很大

### 改进方向

- 使用 PPO/A2C 等连续动作空间的 RL 算法
- 扩展动作空间: 加入止损/止盈参数
- 使用分钟线数据增加训练样本量
- 引入 Multi-Agent RL 同时管理多个策略参数